# Test

# 01 - Configuração SAEC-O&G

Este notebook configura o ambiente para extração CIMO.

**Executar uma vez no início do projeto.**

## Checklist
1. Verificar dependências Python
2. Configurar caminhos
3. Validar API keys
4. Gerar mapping de artigos

## 1. Setup de Paths

In [1]:
import sys
from pathlib import Path

# Detectar diretório do notebook e adicionar src ao path
NOTEBOOK_DIR = Path.cwd()
SYSTEM_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = SYSTEM_DIR / "src"

# Adicionar ao path se necessário
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"System dir: {SYSTEM_DIR}")
print(f"Src dir: {SRC_DIR}")
print(f"Src existe: {SRC_DIR.exists()}")

Notebook dir: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\system\notebooks
System dir: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\system
Src dir: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\system\src
Src existe: True


## 2. Verificar Dependências

In [2]:
import importlib

deps = {
    "dotenv": "python-dotenv",
    "yaml": "pyyaml",
    "pydantic": "pydantic",
    "fitz": "pymupdf",
    "PIL": "pillow",
    "anthropic": "anthropic",
    "openai": "openai",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "tqdm": "tqdm"
}

missing = []
for module, package in deps.items():
    try:
        importlib.import_module(module)
        print(f"✅ {module}")
    except ImportError:
        print(f"❌ {module} - pip install {package}")
        missing.append(package)

if missing:
    print(f"\n⚠️ Execute: pip install {' '.join(missing)}")
else:
    print("\n✅ Todas as dependências instaladas!")

✅ dotenv
✅ yaml
✅ pydantic
✅ fitz
✅ PIL
✅ anthropic
✅ openai
✅ pandas
✅ openpyxl
✅ tqdm

✅ Todas as dependências instaladas!


## 3. Carregar Configuração

In [3]:
from config import paths, llm_config, extraction_config

print("📁 Caminhos configurados:")
print(f"  Project root: {paths.PROJECT_ROOT}")
print(f"  Articles: {paths.ARTICLES}")
print(f"  Extraction: {paths.EXTRACTION}")
print(f"  Outputs: {paths.OUTPUTS}")

# Verificar se pastas existem
print(f"\n📂 Verificação de pastas:")
print(f"  Articles existe: {paths.ARTICLES.exists()}")
print(f"  Extraction existe: {paths.EXTRACTION.exists()}")

📁 Caminhos configurados:
  Project root: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL
  Articles: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\02 T2
  Extraction: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction
  Outputs: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction\outputs

📂 Verificação de pastas:
  Articles existe: True
  Extraction existe: True


In [4]:
# Criar diretórios de output
paths.ensure_dirs()
print("✅ Diretórios de output criados/verificados")
print(f"  Work: {paths.WORK}")
print(f"  YAMLs: {paths.YAMLS}")
print(f"  Consolidated: {paths.CONSOLIDATED}")

✅ Diretórios de output criados/verificados
  Work: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction\outputs\work
  YAMLs: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction\outputs\yamls
  Consolidated: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction\outputs\consolidated


## 4. Verificar APIs

In [5]:
print("🔑 Configuração de APIs:")
print(f"  Two-pass: {llm_config.USE_TWO_PASS}")
print(f"  Primary provider: {llm_config.PRIMARY_PROVIDER}")
print(f"  Max repair attempts: {llm_config.MAX_REPAIR_ATTEMPTS}")

print(f"\n🤖 Modelos:")
print(f"  Anthropic: {llm_config.ANTHROPIC_MODEL}")
print(f"  OpenAI: {llm_config.OPENAI_MODEL}")

print(f"\n🔐 Chaves (mascaradas):")
masked = llm_config.get_masked_keys()
print(f"  Anthropic: {masked['anthropic']}")
print(f"  OpenAI: {masked['openai']}")

🔑 Configuração de APIs:
  Two-pass: True
  Primary provider: ollama
  Max repair attempts: 3

🤖 Modelos:
  Anthropic: claude-opus-4-5
  OpenAI: gpt-5.2-chat-latest

🔐 Chaves (mascaradas):
  Anthropic: CONFIGURADA [OK]
  OpenAI: CONFIGURADA [OK]


In [6]:
# Validar configuração
errors = llm_config.validate()

if errors:
    print("❌ Erros de configuração:")
    for e in errors:
        print(f"  - {e}")
    print("\n⚠️ Edite o arquivo system/.env com suas chaves de API")
    print(f"   Arquivo: {SYSTEM_DIR / '.env'}")
else:
    print("✅ APIs configuradas corretamente!")

✅ APIs configuradas corretamente!


## 5. Gerar Mapping de Artigos

In [7]:
# Listar PDFs disponíveis
pdfs = sorted(paths.ARTICLES.glob("*.pdf"))
print(f"📚 {len(pdfs)} PDFs encontrados em {paths.ARTICLES.name}/")

# Mostrar primeiros 5
for i, pdf in enumerate(pdfs[:5], 1):
    print(f"  {i}. {pdf.name[:60]}...")
if len(pdfs) > 5:
    print(f"  ... e mais {len(pdfs) - 5} arquivos")

📚 38 PDFs encontrados em 02 T2/
  1. A hybrid data envelopment analysis and artificial intelligen...
  2. A Novel Hybrid Model for Vendor Selection in a Supply Chain ...
  3. Adaptive Neural Network for CO2 Reduction A Predictive Model...
  4. An Efficiency-Uncertainty-Based Consensus Optimization Frame...
  5. Applying Machine Learning to the Fuel Theft Problem on Pipel...
  ... e mais 33 arquivos


In [8]:
from config import generate_mapping_csv

# Gerar mapping (não sobrescreve se já existir)
# Para regenerar, use overwrite=True
mapping_path = generate_mapping_csv(
    articles_dir=paths.ARTICLES,
    output_path=paths.MAPPING_CSV,
    overwrite=False  # Mude para True se quiser regenerar
)

[INFO] Mapping já existe: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\Extraction\mapping.csv
[INFO] Use overwrite=True para regenerar


In [9]:
# Visualizar mapping
import pandas as pd

df_mapping = pd.read_csv(paths.MAPPING_CSV)
print(f"\n📊 Mapping: {len(df_mapping)} artigos")
df_mapping.head(10)


📊 Mapping: 38 artigos


,ArtigoID,Arquivo,Processado,Status
0,ART_001,A hybrid data envelopment analysis and artific...,Sim,Aprovado
1,ART_002,A Novel Hybrid Model for Vendor Selection in a...,Sim,Aprovado
2,ART_003,Adaptive Neural Network for CO2 Reduction A Pr...,Sim,Aprovado
3,ART_004,An Efficiency-Uncertainty-Based Consensus Opti...,Sim,Aprovado
4,ART_005,Applying Machine Learning to the Fuel Theft Pr...,Sim,Aprovado
5,ART_006,Assessing_Sustainability_Efforts_and_Greenwash...,Sim,Aprovado
6,ART_007,Best Practices for Building a Modern and Resil...,Sim,Aprovado
7,ART_008,Carbon intensity of global crude oil trading a...,Sim,Aprovado
8,ART_009,China Offshore Intelligent Oilfield Production...,Sim,Aprovado
9,ART_010,Clustering with machine learning and using NDE...,Sim,Aprovado


## 6. Verificar Prompt do Guia

In [10]:
# Verificar se o prompt condensado existe
if paths.GUIA_PROMPT.exists():
    with open(paths.GUIA_PROMPT, "r", encoding="utf-8") as f:
        prompt_content = f.read()
    
    lines = prompt_content.count("\n")
    chars = len(prompt_content)
    
    print(f"✅ Prompt encontrado: {paths.GUIA_PROMPT.name}")
    print(f"   {lines} linhas, {chars} caracteres")
    print(f"   ~{chars // 4} tokens estimados")
else:
    print(f"❌ Prompt não encontrado: {paths.GUIA_PROMPT}")
    print("   Execute o setup completo ou crie o arquivo manualmente")

✅ Prompt encontrado: guia_v3_3_prompt.md
   251 linhas, 9382 caracteres
   ~2345 tokens estimados


## 7. Resumo da Configuração

In [11]:
print("=" * 60)
print("SAEC-O&G - CONFIGURAÇÃO COMPLETA")
print("=" * 60)

# Status geral
api_ok = len(llm_config.validate()) == 0
prompt_ok = paths.GUIA_PROMPT.exists()
mapping_ok = paths.MAPPING_CSV.exists()

print(f"\n📋 Status:")
print(f"   APIs: {'✅' if api_ok else '❌'}")
print(f"   Prompt: {'✅' if prompt_ok else '❌'}")
print(f"   Mapping: {'✅' if mapping_ok else '❌'}")

print(f"\n📊 Artigos:")
print(f"   Total: {len(df_mapping)}")
print(f"   Processados: {len(df_mapping[df_mapping['Processado'] == 'Sim'])}")
print(f"   Pendentes: {len(df_mapping[df_mapping['Processado'] != 'Sim'])}")

print(f"\n💰 Custo estimado:")
print(f"   ~$0.20-0.35 por artigo (Claude + GPT-4o)")
print(f"   ~${len(df_mapping) * 0.25:.2f} total estimado")

print("\n" + "=" * 60)

if api_ok and prompt_ok and mapping_ok:
    print("✅ Tudo pronto! Próximo: 02_Ingestao.ipynb")
else:
    print("⚠️ Corrija os itens marcados com ❌ antes de continuar")

SAEC-O&G - CONFIGURAÇÃO COMPLETA

📋 Status:
   APIs: ✅
   Prompt: ✅
   Mapping: ✅

📊 Artigos:
   Total: 38
   Processados: 25
   Pendentes: 13

💰 Custo estimado:
   ~$0.20-0.35 por artigo (Claude + GPT-4o)
   ~$9.50 total estimado

✅ Tudo pronto! Próximo: 02_Ingestao.ipynb
